In [0]:
# Databricks notebook source

def get_parameters():
    """Retrieve job parameters passed via Databricks Asset Bundles/Job UI."""
    return {
        "source_path": dbutils.widgets.get("source_path"),
        "catalog": dbutils.widgets.get("catalog"),
        "schema": dbutils.widgets.get("schema"),
        "table_name": dbutils.widgets.get("table_name")
    }

def ensure_schema_exists(catalog, schema):
    """Ensure target catalog schema exists."""
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
    print(f"Schema {catalog}.{schema} is ready.")

def create_bronze_table(target_table, source_path):
    """Create Bronze Delta table with enterprise metadata and automatic audit columns."""
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_table} (
        `Transaction unique identifier` STRING COMMENT 'Unique identifier for each transaction record',
        `Price` BIGINT COMMENT 'Transaction sale price in GBP',
        `Date of Transfer` STRING COMMENT 'Timestamp of property transfer',
        `Property Type` STRING COMMENT 'Property type: D=Detached, S=Semi, T=Terraced, F=Flats, O=Other',
        `Old/New` STRING COMMENT 'New build flag: Y=New, N=Established',
        `Duration` STRING COMMENT 'Tenure type: F=Freehold, L=Leasehold',
        `Town/City` STRING COMMENT 'Town or city location',
        `District` STRING COMMENT 'District administrative area',
        `County` STRING COMMENT 'County administrative area',
        `PPDCategory Type` STRING COMMENT 'Price Paid Data category type',
        `Record Status - monthly file only` STRING COMMENT 'Record status code',
        `load_dt` TIMESTAMP DEFAULT current_timestamp() COMMENT 'Audit: Ingestion timestamp into Bronze layer',
        `source` STRING DEFAULT '{source_path}' COMMENT 'Audit: Source file origin path'
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.columnMapping.mode' = 'name',
        'delta.feature.allowColumnDefaults' = 'supported'
    )
    COMMENT 'Bronze transactions table loaded via COPY INTO from landing volume';
    """)
    print(f"Table {target_table} initialized.")

def ingest_data_copy_into(target_table, source_path):
    """Ingest CSV data directly using idempotent COPY INTO."""
    spark.sql(f"""
    COPY INTO {target_table}
    FROM '{source_path}'
    FILEFORMAT = CSV
    FORMAT_OPTIONS ('header' = 'true')
    COPY_OPTIONS ('mergeSchema' = 'true');
    """)
    print(f"Data successfully loaded into {target_table} from {source_path}.")

# COMMAND ----------
# Main Execution Block
if __name__ == "__main__":
    # 1. Retrieve configurations
    params = get_parameters()
    target_table = f"{params['catalog']}.{params['schema']}.{params['table_name']}"
    
    # 2. Setup infrastructure
    ensure_schema_exists(params['catalog'], params['schema'])
    create_bronze_table(target_table, params['source_path'])
    
    # 3. Execute data ingestion
    ingest_data_copy_into(target_table, params['source_path'])